# Preprocessing — SentiMix (SemEval 2020 Task 9)
Loads the Hinglish sentiment dataset, normalizes text, and tokenizes for XLM-R sequence classification.

In [1]:
# Uncomment to install dependencies
# !pip install datasets transformers

In [2]:
import pandas as pd
from datasets import Dataset

c:\Users\Maga\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def parse_conll(filepath):
    # Parse SemEval 2020 Task 9 CoNLL format into a DataFrame.
    # Each sentence starts with a 'meta' line containing the sentiment label,
    # followed by token/language-tag pairs, separated by blank lines.
    
    sentences, labels = [], []
    current_tokens = []
    current_label = None

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("meta"):
                parts = line.split()
                current_label = parts[-1].lower()  # 'positive', 'negative', 'neutral'
            elif line == "":
                if current_tokens and current_label:
                    sentences.append(" ".join(current_tokens))
                    labels.append(current_label)
                    current_tokens = []
                    current_label = None
            else:
                token = line.split()[0]  # first column is the word
                current_tokens.append(token)

    # Catch last sentence if file doesn't end with a blank line
    if current_tokens and current_label:
        sentences.append(" ".join(current_tokens))
        labels.append(current_label)

    return pd.DataFrame({"text": sentences, "label": labels})

In [4]:
# For test since it doesn't have labels
def parse_conll_unlabelled(filepath):
    sentences = []
    current_tokens = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("meta"):
                pass  # no label to extract
            elif line == "":
                if current_tokens:
                    sentences.append(" ".join(current_tokens))
                    current_tokens = []
            else:
                token = line.split()[0]
                current_tokens.append(token)

    if current_tokens:
        sentences.append(" ".join(current_tokens))

    return pd.DataFrame({"text": sentences})

In [ ]:
train_df = parse_conll("data/Hinglish_train_14k_split_conll.txt")
val_df   = parse_conll("data/Hinglish_dev_3k_split_conll.txt")
test_df  = parse_conll_unlabelled("data/Hinglish_test_unlabelled_conll_updated.txt")

valid_labels = {"positive", "negative", "neutral"}
train_df = train_df[train_df["label"].isin(valid_labels)] # some labels are malformed
val_df   = val_df[val_df["label"].isin(valid_labels)]
train_df = train_df[~train_df["text"].str.contains("vist|bolest|vztek")] # some rows are malformed with a different language

raw = {
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df,   preserve_index=False),
    "test": Dataset.from_pandas(test_df,  preserve_index=False)
}

print(f"Train: {len(raw['train'])} | Val: {len(raw['validation'])} | Test: {len(raw['test'])}")

Train: 13994 | Val: 2999 | Test: 3000


In [6]:
# Inspect a few samples
from collections import Counter

print(raw["train"].features)
print(raw["train"][0])
print("\nLabel distribution (train):")
print(Counter(raw["train"]["label"]))

print()
print(raw['test'].features)
print(raw['test'][0])

{'text': Value('large_string'), 'label': Value('large_string')}
{'text': '@ nehantics Haan yaar neha 😔😔 kab karega woh post 😭 Usne na sach mein photoshoot karna chahiye phir woh post karega … https // tco / 5RSlSbZNtt', 'label': 'neutral'}

Label distribution (train):
Counter({'neutral': 5261, 'positive': 4633, 'negative': 4100})

{'text': Value('large_string')}
{'text': '@ 454dkhan @ Heisunberg _ Agr kse ko itni importantce chaeay ni tou ðŸ˜…'}


In [7]:
import re

def normalize_hinglish(text):
    text = str(text).lower()

    # Remove URLs, mentions, hashtags FIRST (before char removal)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'#\S+', '', text)

    # Remove special characters except basic punctuation
    text = re.sub(r'[^\w\s.,!?]', '', text)

    # Collapse repeated characters (e.g. 'soooo' → 'soo')
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Collapse extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Define ALL replacements in one dictionary (fixes original bug)
    replacements = {
        # Romanized Hindi spelling variants
        "bohot": "bahut",
        "bohut": "bahut",
        "bahot": "bahut",
        "kyaa":  "kya",
        "kyu":   "kyun",
        "plzz":  "please",
        "plz":   "please",
        # Hindi number words
        "ek":    "one",
        "do":    "two",
        "teen":  "three",
        # Internet slang
        "lol":   "laughing",
        "lmao":  "laughing",
        "omg":   "oh my god",
        "tbh":   "to be honest",
        "idk":   "i don't know",
        "btw":   "by the way",
        "brb":   "be right back",
    }

    for k, v in replacements.items():
        text = re.sub(rf'\b{k}\b', v, text)  # word-boundary match avoids partial replacements

    return text

# Quick check
print(normalize_hinglish("yaar bohot accha tha!! plz btw lol http://t.co/xyz @user"))

yaar bahut accha tha!! please by the way laughing


In [8]:
TEXT_COL = "text"

for split in ["train", "validation", "test"]:
    raw[split] = raw[split].map(
        lambda x: {TEXT_COL: normalize_hinglish(x[TEXT_COL])}
    )

print("Sample after normalization:")
print(raw["train"][0])

Map: 100%|██████████| 3000/3000 [00:00<00:00, 12688.44 examples/s]

Sample after normalization:
{'text': 'nehantics haan yaar neha kab karega woh post usne na sach mein photoshoot karna chahiye phir woh post karega tco 5rslsbzntt', 'label': 'neutral'}


In [9]:
from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Loaded tokenizer: {MODEL_NAME}")

Loaded tokenizer: xlm-roberta-base


In [10]:
# Map string labels to integers for classification
LABEL2ID = {"positive": 0, "negative": 1, "neutral": 2}

def tokenize(examples):
    tokenized = tokenizer(
        examples[TEXT_COL],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    tokenized["label"] = [LABEL2ID[l.strip().lower()] for l in examples["label"]]
    return tokenized

def tokenize_test(examples):
    return tokenizer(
        examples[TEXT_COL],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

for split in ["train", "validation"]:
    raw[split] = raw[split].map(
        tokenize,
        batched=True,
        remove_columns=["text"]
    )
    raw[split].set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Test has no label column
raw["test"] = raw["test"].map(tokenize_test, batched=True, remove_columns=["text"])
raw["test"].set_format(type="torch", columns=["input_ids", "attention_mask"])

print("Train sample keys:", raw["train"][0].keys())
print(f"Splits — train: {len(raw['train'])}, val: {len(raw['validation'])}, test: {len(raw['test'])}")

Map: 100%|██████████| 3000/3000 [00:00<00:00, 26995.41 examples/s]

Train sample keys: dict_keys(['label', 'input_ids', 'attention_mask'])
Splits — train: 13994, val: 2999, test: 3000


In [11]:
from datasets import Value

for split in ["train", "validation"]:
    raw[split] = raw[split].cast_column("label", Value("int64"))

Casting the dataset: 100%|██████████| 2999/2999 [00:00<00:00, 1084652.73 examples/s]


In [ ]:
# Make sure that label is an int or tensor int
sample = raw["train"][0]
print("Label value:", sample["label"])
print("Label type:", type(sample["label"]))
print("Input IDs length:", len(sample["input_ids"]))

Label value: tensor(2)
Label type: <class 'torch.Tensor'>
Input IDs length: 128


In [18]:
raw["train"].save_to_disk("train_tok")
raw["validation"].save_to_disk("val_tok")
raw["test"].save_to_disk("test_tok")

print("Saved: train_tok, val_tok, test_tok")

Saving the dataset (1/1 shards): 100%|██████████| 3000/3000 [00:00<00:00, 741873.24 examples/s]

Saved: train_tok, val_tok, test_tok
